In [2]:
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
import os
import random

In [3]:
def setup_seed(seed):
    np.random.seed(seed)  
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1' 
setup_seed(32) 
random.seed(0)

In [4]:
# 1. 加载模型
model = tf.keras.models.load_model(
    './Model_weights_2026_3_15/(2026_3_15).4.779379.h5' #CNN调参重新训练后可以进行替换
)

2026-03-15 14:01:59.569292: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-15 14:02:02.046728: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22302 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:1d:00.0, compute capability: 8.6


In [5]:
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_12 (Conv2D)          (None, 64, 64, 8)         224       
                                                                 
 batch_normalization_12 (Bat  (None, 64, 64, 8)        32        
 chNormalization)                                                
                                                                 
 average_pooling2d_12 (Avera  (None, 32, 32, 8)        0         
 gePooling2D)                                                    
                                                                 
 conv2d_13 (Conv2D)          (None, 32, 32, 16)        1168      
                                                                 
 batch_normalization_13 (Bat  (None, 32, 32, 16)       64        
 chNormalization)                                                
                                                      

In [6]:
# ==============================
# 2. TSNE坐标（训练阶段已经得到）
# ==============================

data_new = np.array([
[-2.2956204, 11.983552],
[8.972184, 7.842322],
[1.8764849,-4.529769],
[-12.906775,3.3454704],
[-2.1228542,-47.829395],
[-25.717352,12.619746]
])

x_coord = data_new[:,0]
y_coord = data_new[:,1]

x_min,y_min,x_max,y_max = min(x_coord),min(y_coord),max(x_coord),max(y_coord)

x_pixel = (1 + 8*((x_coord-x_min)/(x_max-x_min)))
y_pixel = (1 + 8*((y_coord-y_min)/(y_max-y_min)))

round_x_pixel = np.round(x_pixel).astype(int)
round_y_pixel = np.round(y_pixel).astype(int)


In [7]:
data_max = np.array([0.930982, 0.803183, 0.952556, 0.075975, 0.928571, 0.727382], dtype=np.float32) #前面基于数据集统计的每个特征最大值最小值
data_min = np.array([0.000000, 0.108040, 0.383712, 0.000287, 0.128571, 0.093170], dtype=np.float32)

# =========================
# 3. 单条样本归一化
# =========================
def normalize_sample(sample):
    sample = np.array(sample, dtype=np.float32)

    sample_norm = (sample - data_min) / (data_max - data_min)

    return sample_norm

In [8]:
# ==============================
# 4. 单条样本 -> 图像
# ==============================
def sample_to_image(sample):

    zero = np.zeros((9,9))

    k = 0
    for l,m in zip(round_x_pixel,round_y_pixel):

        if k < sample.shape[0]:
            zero[m-1][l-1] = sample[k]
            k += 1

    # heatmap -> image
    fig = plt.figure(figsize=(5,5))
    ax = sns.heatmap(
        zero,
        center=0,
        vmin=0,
        vmax=1,
        cbar=False
    )

    ax.set(xticklabels=[])
    ax.set(yticklabels=[])

    ax.tick_params(left=False,bottom=False)

    plt.gca().xaxis.set_major_locator(plt.NullLocator())
    plt.gca().yaxis.set_major_locator(plt.NullLocator())

    plt.subplots_adjust(top=1,bottom=0,right=1,left=0,hspace=0,wspace=0)
    plt.margins(0,0)

    # 转成numpy图像
    fig.canvas.draw()
    img = np.frombuffer(fig.canvas.tostring_rgb(),dtype=np.uint8)
    img = img.reshape(fig.canvas.get_width_height()[::-1]+(3,))

    plt.close()

    return img

In [15]:
# ==============================
# 5. 推理函数
# ==============================
def predict_single(sample):
    # 1) 归一化
    sample_norm = normalize_sample(sample)

    # 2) 转图像
    img = sample_to_image(sample_norm)

    # 3) 翻转
    img = cv2.flip(img, 0)

    # 4) resize
    img = cv2.resize(img, (64, 64))

    # 5) 归一化到[0,1]
    img = img.astype("float32") / 255.0

    # 6) 增加batch维度
    img = np.expand_dims(img, axis=0)
    
    # 7) 预测
    pred = model.predict(img, verbose=0)

    return pred

In [28]:
sample = np.array([2.18,9.48,31.34,0.53,90.00,17.51])

In [29]:
pred = predict_single(sample)

print("Prediction:",pred)

Prediction: [[22.246765 42.094833]]
